# SU(4) exceptional-rank corpus and resolvent audit
Run the single code cell. Upload the full symbolic source bundle when prompted.

In [ ]:
#!/usr/bin/env python3
"""
SU(4) exceptional-rank O(y^4) corpus enumerator and resolvent-impact audit.

This is the first exact SU(4) stage. It reuses the verified group-independent
Stage-0 geometry and the stable-rank Stage-1 sign basis, replacing only

    local charge == 0

by the exact SU(4) N-ality condition

    local charge == 0 mod 4.

Unlike SU(6), SU(4) determinant sectors can occur at the three resolvent cuts.
This script therefore does not attempt the contraction. It enumerates the full
exceptional word/sign corpus and classifies, exactly:

  * unchanged stable-rank regression;
  * new and mixed SU(4) words;
  * final local Haar families (4,0), (0,4), (5,1), (1,5);
  * every cut at which a pure determinant channel Lambda^4 V occurs;
  * canonical local signatures needed by the finite-rank contraction engine.

Input:
  Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE_2026-06-14_V2*.zip

Outputs:
  SU4_EXCEPTIONAL_ENUMERATOR_V1/SU4_EXCEPTIONAL_ENUMERATOR_V1.json
  SU4_EXCEPTIONAL_ENUMERATOR_V1/SU4_EXCEPTIONAL_ENUMERATOR_V1.md
  SU4_EXCEPTIONAL_ENUMERATOR_V1/y4_su4_ordered_words.json.gz
  SU4_EXCEPTIONAL_ENUMERATOR_V1/y4_su4_exceptional_only_words.json.gz
  SU4_EXCEPTIONAL_ENUMERATOR_V1/y4_su4_local_signature_catalog.json
  SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE.zip
"""
from __future__ import annotations

import gzip
import hashlib
import importlib.util
import itertools
import json
import os
import re
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Iterable

VERSION = "2026-06-14-su4-exceptional-enumerator-v1"
N = 4
BASE = Path("/content") if Path("/content").exists() else Path("/mnt/data")
OUT = BASE / "SU4_EXCEPTIONAL_ENUMERATOR_V1"
EXTRACT = OUT / "extracted"
OUT.mkdir(parents=True, exist_ok=True)
EXTRACT.mkdir(parents=True, exist_ok=True)

PREFERRED_BUNDLE_GLOB = "Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE*.zip"
REQUIRED_STAGE1 = "y4_sun_stable_rank_stage1.py"
REQUIRED_WORDS = "y4_sun_stable_ordered_words.json.gz"


def gate(name: str, cond: bool, detail: str = "") -> None:
    status = "PASS" if cond else "FAIL"
    print(f"{status:4s} {name:88s} {detail}")
    if not cond:
        raise AssertionError(f"{name}: {detail}")


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path: Path, payload: Any) -> str:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    return sha256(path)


def write_json_gz(path: Path, payload: Any) -> str:
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    with path.open("wb") as raw_file:
        with gzip.GzipFile(filename="", mode="wb", compresslevel=9, fileobj=raw_file, mtime=0) as f:
            f.write(raw)
    return sha256(path)


def read_json_gz(path: Path) -> Any:
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)


def safe_extract(zpath: Path, dest: Path) -> list[Path]:
    dest.mkdir(parents=True, exist_ok=True)
    root = dest.resolve()
    with zipfile.ZipFile(zpath) as zf:
        for info in zf.infolist():
            target = (dest / info.filename).resolve()
            if target != root and not str(target).startswith(str(root) + os.sep):
                raise ValueError(f"unsafe ZIP member: {info.filename}")
        zf.extractall(dest)
        return [dest / i.filename for i in zf.infolist() if not i.is_dir()]


def recursive_extract(archives: Iterable[Path], max_depth: int = 4) -> list[dict[str, Any]]:
    queue = [(Path(p), 0) for p in archives]
    seen: set[str] = set()
    records: list[dict[str, Any]] = []
    while queue:
        zp, depth = queue.pop(0)
        if depth > max_depth or not zp.is_file():
            continue
        try:
            h = sha256(zp)
        except Exception:
            continue
        if h in seen:
            continue
        seen.add(h)
        label = re.sub(r"[^A-Za-z0-9_.-]+", "_", zp.stem)[:90]
        dest = EXTRACT / f"d{depth}_{label}_{h[:10]}"
        try:
            files = safe_extract(zp, dest)
            records.append({
                "archive": str(zp), "sha256": h, "depth": depth,
                "destination": str(dest), "file_count": len(files), "status": "ok",
            })
            for p in files:
                if p.suffix.lower() == ".zip":
                    queue.append((p, depth + 1))
        except Exception as exc:
            records.append({
                "archive": str(zp), "sha256": h, "depth": depth,
                "status": "error", "error": repr(exc),
            })
    return records


def find_unique(name: str, roots: Iterable[Path]) -> list[Path]:
    found: dict[str, Path] = {}
    for root in roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob(name):
                if p.is_file():
                    found[sha256(p)] = p
        except Exception:
            pass
    return sorted(found.values(), key=lambda p: str(p))


def find_glob(pattern: str, roots: Iterable[Path]) -> list[Path]:
    found: dict[str, Path] = {}
    for root in roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob(pattern):
                if p.is_file():
                    found[sha256(p)] = p
        except Exception:
            pass
    return sorted(found.values(), key=lambda p: str(p))


def upload_if_needed() -> list[Path]:
    roots = [BASE]
    candidates = find_glob(PREFERRED_BUNDLE_GLOB, roots)
    candidates += find_unique(REQUIRED_STAGE1, roots)
    candidates += find_unique(REQUIRED_WORDS, roots)
    if candidates:
        return candidates
    if not Path("/content").exists():
        return []
    try:
        from google.colab import files as colab_files  # type: ignore
    except Exception:
        return []
    print("\nUPLOAD REQUIRED — select the full symbolic source bundle:")
    print("  Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE_2026-06-14_V2*.zip")
    uploaded = colab_files.upload()
    saved: list[Path] = []
    for name, data in uploaded.items():
        target = Path("/content") / Path(name).name
        target.write_bytes(data)
        saved.append(target)
        print(f"saved {len(data):,} bytes -> {target}")
    return saved


def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def bit_indices(mask: int) -> list[int]:
    return [i for i in range(64) if (mask >> i) & 1]


def key_from_record(record: dict[str, Any]):
    return (
        tuple(tuple(int(x) for x in p) for p in record["ordered_insertions"]),
        tuple(int(x) for x in record["output"]),
    )


def assignments_from_record(record: dict[str, Any]) -> set[tuple[int, ...]]:
    return {tuple(int(x) for x in s) for s in record["exact_balance_assignments"]}


def c_representative(signs: tuple[int, ...]) -> tuple[int, ...]:
    conjugate = tuple(-x for x in signs)
    return min(signs, conjugate)


def signature_representative(sig: tuple[int, ...]) -> tuple[int, ...]:
    conjugate = tuple(-x for x in sig)
    return min(sig, conjugate)


def main() -> None:
    started = time.time()
    print("=" * 116)
    print("SU(4) O(y^4) EXCEPTIONAL-RANK CORPUS ENUMERATOR + RESOLVENT AUDIT")
    print("=" * 116)
    print("version :", VERSION)
    print("output  :", OUT)
    print("hardware: CPU exact combinatorics; GPU not used")

    initial = upload_if_needed()
    archives = [p for p in initial if p.suffix.lower() == ".zip"]
    archives += find_glob(PREFERRED_BUNDLE_GLOB, [BASE])
    extraction = recursive_extract(archives)
    roots = [BASE, EXTRACT]

    stage1_paths = find_unique(REQUIRED_STAGE1, roots)
    words_paths = find_unique(REQUIRED_WORDS, roots)
    gate("stable-rank Stage-1 source located", bool(stage1_paths), str(stage1_paths[:3]))
    gate("4,171-word stable archive located", bool(words_paths), str(words_paths[:3]))

    st_path = stage1_paths[0]
    words_path = words_paths[0]
    st = load_module("y4_sun_stable_rank_stage1_su4_enum", st_path)

    required_names = [
        "locate_complete_source", "decode_stage1", "load_module_from_source",
        "ensure_stage0_supports", "rows_for", "SIGNS", "FULL_MASK",
    ]
    missing = [name for name in required_names if not hasattr(st, name)]
    gate("stable Stage-1 runtime contract available", not missing, str(missing))
    gate("stable sign basis contains 64 assignments", len(st.SIGNS) == 64, str(len(st.SIGNS)))

    complete_source = st.locate_complete_source(None)
    stage0_source, stage1_source = st.decode_stage1(complete_source)
    stage0 = st.load_module_from_source("y4_stage0_su4_enum", stage0_source)
    stage1_local = st.load_module_from_source("y4_stage1_su4_enum", stage1_source)
    support_path = Path(st.ensure_stage0_supports(stage0, stage0_source, None))
    support_payload = read_json_gz(support_path)
    supports = [
        tuple(tuple(int(x) for x in p) for p in record)
        for record in support_payload["supports"]
    ]
    gate("Stage-0 connected support count", len(supports) == 182440, f"{len(supports):,}")

    signs_basis = tuple(tuple(int(x) for x in s) for s in st.SIGNS)
    full_mask = int(st.FULL_MASK)
    gate("FULL_MASK covers all 64 assignments", full_mask == (1 << 64) - 1, hex(full_mask))

    row_cache_stable: dict[tuple[int, ...], int] = {}
    row_cache_su4: dict[tuple[int, ...], int] = {}
    max_local_degree = 0

    def row_mask(row: tuple[int, ...], *, su4: bool) -> int:
        nonlocal max_local_degree
        cache = row_cache_su4 if su4 else row_cache_stable
        if row in cache:
            return cache[row]
        max_local_degree = max(max_local_degree, sum(abs(int(x)) for x in row))
        mask = 0
        for index, signs in enumerate(signs_basis):
            charge = sum(int(row[j]) * int(signs[j]) for j in range(6))
            ok = (charge % N == 0) if su4 else (charge == 0)
            if ok:
                mask |= 1 << index
        cache[row] = mask
        return mask

    def mask_for(word, output, *, su4: bool) -> int:
        mask = full_mask
        for row0 in st.rows_for(stage0, word, output):
            row = tuple(int(x) for x in row0)
            mask &= row_mask(row, su4=su4)
            if mask == 0:
                break
        return mask

    stable_survivors: dict[Any, int] = {}
    su4_survivors: dict[Any, int] = {}
    candidate_pairs = 0

    for index, multiset in enumerate(supports, start=1):
        for output in stage0.candidate_outputs(multiset):
            candidate_pairs += 1
            stable_mask = mask_for(multiset, output, su4=False)
            su4_mask = mask_for(multiset, output, su4=True)
            if stable_mask:
                stable_survivors[stage0.canonical_support_output(multiset, output)] = 1
            if su4_mask:
                su4_survivors[stage0.canonical_support_output(multiset, output)] = 1
            if stable_mask & ~su4_mask:
                raise AssertionError("stable assignment absent from SU(4) sector")
        if index % 40000 == 0:
            print(
                f"[geometry] supports={index:,}/{len(supports):,} "
                f"stable_classes={len(stable_survivors):,} su4_classes={len(su4_survivors):,}",
                flush=True,
            )

    gate("candidate support/output pair count", candidate_pairs == 895524, f"{candidate_pairs:,}")
    gate("stable support/output classes reproduced", len(stable_survivors) == 439, str(len(stable_survivors)))
    gate("stable support classes are an SU(4) subset",
         set(stable_survivors).issubset(set(su4_survivors)),
         f"stable={len(stable_survivors)} su4={len(su4_survivors)}")

    def ordered_keys_from(survivors: dict[Any, int]) -> set[Any]:
        keys = set()
        for multiset, output in survivors:
            for word in set(itertools.permutations(multiset)):
                keys.add(stage0.canonical_ordered_transition(word, output))
        return keys

    stable_ordered_keys = ordered_keys_from(stable_survivors)
    su4_ordered_keys = ordered_keys_from(su4_survivors)
    gate("stable ordered-key candidates are an SU(4) subset",
         stable_ordered_keys.issubset(su4_ordered_keys),
         f"stable={len(stable_ordered_keys)} su4={len(su4_ordered_keys)}")

    stable_ordered: dict[Any, int] = {}
    su4_ordered: dict[Any, int] = {}
    exceptional_ordered: dict[Any, int] = {}

    for idx, (word, output) in enumerate(sorted(su4_ordered_keys), start=1):
        stable_mask = mask_for(word, output, su4=False)
        su4_mask = mask_for(word, output, su4=True)
        exceptional_mask = su4_mask & ~stable_mask
        if stable_mask:
            stable_ordered[(word, output)] = stable_mask
        if su4_mask:
            su4_ordered[(word, output)] = su4_mask
        if exceptional_mask:
            exceptional_ordered[(word, output)] = exceptional_mask
        if idx % 10000 == 0:
            print(
                f"[ordered] keys={idx:,}/{len(su4_ordered_keys):,} "
                f"stable={len(stable_ordered):,} exceptional={len(exceptional_ordered):,}",
                flush=True,
            )

    stable_assignments = sum(mask.bit_count() for mask in stable_ordered.values())
    gate("stable ordered words reproduced", len(stable_ordered) == 4171, str(len(stable_ordered)))
    gate("stable sign assignments reproduced", stable_assignments == 33500, f"{stable_assignments:,}")
    gate("stable charge-conjugation orbits reproduced", stable_assignments // 2 == 16750,
         f"{stable_assignments // 2:,}")

    archive_payload = read_json_gz(words_path)
    archive_records = archive_payload["words"]
    gate("reference archive record count", len(archive_records) == 4171, str(len(archive_records)))
    archive_map = {key_from_record(r): assignments_from_record(r) for r in archive_records}
    computed_map = {
        key: {signs_basis[i] for i in bit_indices(mask)}
        for key, mask in stable_ordered.items()
    }
    gate("stable ordered-word key set exactly matches reference archive",
         set(computed_map) == set(archive_map),
         f"computed={len(computed_map)} archive={len(archive_map)}")
    mismatch_keys = [k for k in archive_map if archive_map[k] != computed_map.get(k)]
    gate("all stable assignment sets exactly match reference archive",
         not mismatch_keys, f"mismatches={len(mismatch_keys)}")

    exceptional_assignment_count = sum(mask.bit_count() for mask in exceptional_ordered.values())
    gate("exceptional assignment count is even under charge conjugation",
         exceptional_assignment_count % 2 == 0, str(exceptional_assignment_count))

    new_word_keys = set(su4_ordered) - set(stable_ordered)
    mixed_word_keys = set(exceptional_ordered) & set(stable_ordered)
    pure_exceptional_keys = set(exceptional_ordered) - set(stable_ordered)
    gate("new SU(4) words are exactly pure exceptional words",
         new_word_keys == pure_exceptional_keys,
         f"new={len(new_word_keys)} pure={len(pure_exceptional_keys)}")

    archive_id = {key_from_record(r): r["ordered_id"] for r in archive_records}

    final_family_hist: Counter[tuple[int, int]] = Counter()
    final_charge_hist: Counter[int] = Counter()
    exceptional_links_per_assignment: Counter[int] = Counter()
    cut_det_hist: Counter[int] = Counter()
    cut_det_family_hist: Counter[tuple[int, int, int]] = Counter()
    assignments_with_resolvent_det = 0
    assignments_final_only = 0
    signature_occurrences: Counter[tuple[int, ...]] = Counter()
    canonical_signature_occurrences: Counter[tuple[int, ...]] = Counter()
    signature_cut_profiles: dict[tuple[int, ...], tuple[int, int, int]] = {}
    bad: list[dict[str, Any]] = []
    exceptional_records: list[dict[str, Any]] = []

    for record_index, ((word, output), exceptional_mask) in enumerate(
        sorted(exceptional_ordered.items()), start=1
    ):
        factors = (stage0.ROOT,) + tuple(word) + (output,)
        links = sorted({
            link
            for plaquette in factors
            for link, _incidence in stage1_local.boundary(plaquette)
        })
        assignment_records = []
        for sign_index in bit_indices(exceptional_mask):
            signs = signs_basis[sign_index]
            local_records = []
            has_resolvent_det = False
            exceptional_link_count = 0
            for link in links:
                tokens = tuple(int(x) for x in stage1_local.factor_tokens(factors, signs, link))
                nf = sum(t == 1 for t in tokens)
                na = sum(t == -1 for t in tokens)
                charge = nf - na
                if charge == 0:
                    continue
                exceptional_link_count += 1
                final_family_hist[(nf, na)] += 1
                final_charge_hist[charge] += 1
                signature_occurrences[tokens] += 1
                canonical_signature_occurrences[signature_representative(tokens)] += 1
                cut_charges = tuple(sum(tokens[:cut + 1]) for cut in (1, 2, 3))
                signature_cut_profiles[tokens] = cut_charges
                det_cuts = []
                for cut_number, qcut in zip((1, 2, 3), cut_charges):
                    if abs(qcut) == 4:
                        prefix = tokens[:cut_number + 1]
                        pnf = sum(t == 1 for t in prefix)
                        pna = sum(t == -1 for t in prefix)
                        det_cuts.append(cut_number)
                        cut_det_hist[cut_number] += 1
                        cut_det_family_hist[(cut_number, pnf, pna)] += 1
                        has_resolvent_det = True
                    elif abs(qcut) > 4:
                        bad.append({
                            "reason": "unexpected_prefix_charge",
                            "tokens": tokens, "cut": cut_number, "charge": qcut,
                        })
                if charge not in (-4, 4):
                    bad.append({
                        "reason": "unexpected_final_charge",
                        "tokens": tokens, "charge": charge,
                    })
                if (nf, na) not in {(4, 0), (0, 4), (5, 1), (1, 5)}:
                    bad.append({
                        "reason": "unexpected_final_family",
                        "tokens": tokens, "family": (nf, na),
                    })
                local_records.append({
                    "link": repr(link),
                    "tokens": list(tokens),
                    "family": [nf, na],
                    "charge": charge,
                    "cut_charges": list(cut_charges),
                    "determinant_cuts": det_cuts,
                })
            if exceptional_link_count == 0:
                bad.append({
                    "reason": "exceptional_assignment_without_exceptional_link",
                    "word": word, "output": output, "signs": signs,
                })
            exceptional_links_per_assignment[exceptional_link_count] += 1
            if has_resolvent_det:
                assignments_with_resolvent_det += 1
            else:
                assignments_final_only += 1
            assignment_records.append({
                "signs": list(signs),
                "c_representative": list(c_representative(signs)),
                "has_resolvent_determinant": has_resolvent_det,
                "exceptional_link_count": exceptional_link_count,
                "local_records": local_records,
            })
        exceptional_records.append({
            "su4_exceptional_id": f"S4X4-{record_index:05d}",
            "stable_ordered_id": archive_id.get((word, output)),
            "root": list(stage0.ROOT),
            "ordered_insertions": [list(p) for p in word],
            "output": list(output),
            "word_sector": (
                "mixed_stable_and_exceptional"
                if (word, output) in stable_ordered
                else "exceptional_only"
            ),
            "exceptional_assignment_count": exceptional_mask.bit_count(),
            "assignments": assignment_records,
        })

    gate("every exceptional assignment contains a nonzero SU(4) determinant-family link",
         not bad, f"bad={len(bad)}")
    gate("only final local charges +4 and -4 occur",
         set(final_charge_hist).issubset({-4, 4}), str(dict(final_charge_hist)))
    gate("only SU(4) exceptional final families occur",
         set(final_family_hist).issubset({(4, 0), (0, 4), (5, 1), (1, 5)}),
         str(dict(final_family_hist)))
    gate("resolvent determinant prefixes are pure four-strand sectors",
         all((pnf, pna) in {(4, 0), (0, 4)}
             for _cut, pnf, pna in cut_det_family_hist),
         str(dict(cut_det_family_hist)))
    gate("maximum final local tensor degree remains six", max_local_degree <= 6, str(max_local_degree))
    gate("all exceptional assignments classified",
         assignments_with_resolvent_det + assignments_final_only == exceptional_assignment_count,
         f"{assignments_with_resolvent_det}+{assignments_final_only}={exceptional_assignment_count}")

    all_records = []
    for idx, (key, su4_mask) in enumerate(sorted(su4_ordered.items()), start=1):
        word, output = key
        stable_mask = stable_ordered.get(key, 0)
        exceptional_mask = exceptional_ordered.get(key, 0)
        all_records.append({
            "su4_ordered_id": f"S4N4-{idx:05d}",
            "stable_ordered_id": archive_id.get(key),
            "root": list(stage0.ROOT),
            "ordered_insertions": [list(p) for p in word],
            "output": list(output),
            "su4_assignment_count": su4_mask.bit_count(),
            "stable_assignment_count": stable_mask.bit_count(),
            "exceptional_assignment_count": exceptional_mask.bit_count(),
            "stable_assignments": [list(signs_basis[i]) for i in bit_indices(stable_mask)],
            "exceptional_assignments": [list(signs_basis[i]) for i in bit_indices(exceptional_mask)],
            "word_sector": (
                "mixed_stable_and_exceptional" if stable_mask and exceptional_mask
                else "stable_only" if stable_mask
                else "exceptional_only"
            ),
        })

    all_path = OUT / "y4_su4_ordered_words.json.gz"
    exceptional_path = OUT / "y4_su4_exceptional_only_words.json.gz"
    catalog_path = OUT / "y4_su4_local_signature_catalog.json"

    all_sha = write_json_gz(all_path, {
        "meta": {
            "version": VERSION, "rank": N,
            "criterion": "local charge is 0 modulo 4",
            "stable_reference": str(words_path),
            "stable_reference_sha256": sha256(words_path),
        },
        "counts": {
            "ordered_words": len(all_records),
            "stable_words": len(stable_ordered),
            "exceptional_bearing_words": len(exceptional_ordered),
            "new_exceptional_only_words": len(new_word_keys),
            "mixed_words": len(mixed_word_keys),
            "stable_assignments": stable_assignments,
            "exceptional_assignments": exceptional_assignment_count,
        },
        "words": all_records,
    })
    exceptional_sha = write_json_gz(exceptional_path, {
        "meta": {
            "version": VERSION, "rank": N,
            "scope": "assignments absent from the N>=7 exact-balance corpus",
            "criterion": "at least one final local charge is +4 or -4",
        },
        "counts": {
            "exceptional_bearing_words": len(exceptional_records),
            "new_exceptional_only_words": len(new_word_keys),
            "mixed_words": len(mixed_word_keys),
            "exceptional_assignments": exceptional_assignment_count,
            "exceptional_charge_conjugation_orbits": exceptional_assignment_count // 2,
            "assignments_with_resolvent_determinant": assignments_with_resolvent_det,
            "assignments_final_only": assignments_final_only,
        },
        "words": exceptional_records,
    })

    catalog_rows = []
    for sig, count in sorted(signature_occurrences.items()):
        nf = sig.count(1)
        na = sig.count(-1)
        catalog_rows.append({
            "signature": list(sig),
            "canonical_under_C": list(signature_representative(sig)),
            "occurrences": count,
            "family": [nf, na],
            "final_charge": nf - na,
            "cut_charges": list(signature_cut_profiles[sig]),
            "determinant_cuts": [
                cut for cut, q in zip((1, 2, 3), signature_cut_profiles[sig]) if abs(q) == 4
            ],
        })
    catalog_sha = write_json(catalog_path, {
        "meta": {"version": VERSION, "rank": N},
        "counts": {
            "oriented_signatures": len(signature_occurrences),
            "C_canonical_signatures": len(canonical_signature_occurrences),
            "total_occurrences": sum(signature_occurrences.values()),
        },
        "final_family_histogram": {
            f"{a},{b}": v for (a, b), v in sorted(final_family_hist.items())
        },
        "cut_determinant_histogram": {str(k): v for k, v in sorted(cut_det_hist.items())},
        "cut_determinant_family_histogram": {
            f"cut={cut};family=({a},{b})": v
            for (cut, a, b), v in sorted(cut_det_family_hist.items())
        },
        "signatures": catalog_rows,
    })

    next_stage = (
        "A finite-rank SU(4) local library is required because determinant channels occur "
        "inside resolvent cuts. Treat every pure four-strand prefix as the SU(4) singlet "
        "Lambda^4 V with C2=0, and implement final epsilon/delta Haar tensors for the "
        "(4,0),(0,4),(5,1),(1,5) families. Preserve the stable 4,171-word contraction unchanged."
        if assignments_with_resolvent_det
        else
        "No determinant channel occurs in a resolvent cut; contract only the final exceptional Haar nodes."
    )

    summary = {
        "version": VERSION,
        "status": "PASS",
        "inputs": {
            "stage1_source": str(st_path),
            "stage1_source_sha256": sha256(st_path),
            "stable_words": str(words_path),
            "stable_words_sha256": sha256(words_path),
            "complete_source": str(complete_source),
            "supports": str(support_path),
            "supports_sha256": sha256(support_path),
            "extraction": extraction,
        },
        "stable_regression": {
            "support_output_classes": len(stable_survivors),
            "ordered_words": len(stable_ordered),
            "sign_assignments": stable_assignments,
            "charge_conjugation_orbits": stable_assignments // 2,
            "archive_key_set_exact": True,
            "archive_assignment_sets_exact": True,
        },
        "su4": {
            "support_output_classes": len(su4_survivors),
            "ordered_words": len(su4_ordered),
            "total_assignments": sum(mask.bit_count() for mask in su4_ordered.values()),
            "exceptional_bearing_words": len(exceptional_ordered),
            "new_exceptional_only_words": len(new_word_keys),
            "mixed_words": len(mixed_word_keys),
            "exceptional_assignments": exceptional_assignment_count,
            "exceptional_charge_conjugation_orbits": exceptional_assignment_count // 2,
            "assignments_with_resolvent_determinant": assignments_with_resolvent_det,
            "assignments_final_only": assignments_final_only,
            "final_family_histogram": {
                f"({a},{b})": v for (a, b), v in sorted(final_family_hist.items())
            },
            "final_charge_histogram": {str(k): v for k, v in sorted(final_charge_hist.items())},
            "exceptional_links_per_assignment": {
                str(k): v for k, v in sorted(exceptional_links_per_assignment.items())
            },
            "cut_determinant_histogram": {str(k): v for k, v in sorted(cut_det_hist.items())},
            "oriented_local_signatures": len(signature_occurrences),
            "C_canonical_local_signatures": len(canonical_signature_occurrences),
            "max_local_degree": max_local_degree,
        },
        "outputs": {
            "all_words": str(all_path), "all_words_sha256": all_sha,
            "exceptional_words": str(exceptional_path), "exceptional_words_sha256": exceptional_sha,
            "signature_catalog": str(catalog_path), "signature_catalog_sha256": catalog_sha,
        },
        "next_stage": next_stage,
        "elapsed_seconds": time.time() - started,
    }

    json_path = OUT / "SU4_EXCEPTIONAL_ENUMERATOR_V1.json"
    json_sha = write_json(json_path, summary)

    md = f"""# SU(4) exceptional-rank O(y^4) corpus

**Status:** PASS  
**Version:** `{VERSION}`

## Stable-rank regression

- support/output classes: **{len(stable_survivors):,}**
- ordered words: **{len(stable_ordered):,}**
- sign assignments: **{stable_assignments:,}**
- charge-conjugation orbits: **{stable_assignments // 2:,}**
- archive keys and assignment sets: **exact match**

## SU(4) extension

- support/output classes: **{len(su4_survivors):,}**
- ordered words: **{len(su4_ordered):,}**
- exceptional-bearing words: **{len(exceptional_ordered):,}**
- new exceptional-only words: **{len(new_word_keys):,}**
- mixed stable/exceptional words: **{len(mixed_word_keys):,}**
- exceptional assignments: **{exceptional_assignment_count:,}**
- exceptional C-orbits: **{exceptional_assignment_count // 2:,}**

Final exceptional family histogram:

```text
{dict(sorted(final_family_hist.items()))}
```

Assignments with a determinant channel in at least one resolvent cut:

```text
{assignments_with_resolvent_det:,}
```

Assignments whose determinant structure occurs only in the final Haar integral:

```text
{assignments_final_only:,}
```

Cut histogram for pure `Lambda^4 V` determinant channels:

```text
{dict(sorted(cut_det_hist.items()))}
```

## Consequence

{next_stage}

## Outputs

- `{all_path.name}`
- `{exceptional_path.name}`
- `{catalog_path.name}`
- `{json_path.name}`
"""
    md_path = OUT / "SU4_EXCEPTIONAL_ENUMERATOR_V1.md"
    md_path.write_text(md, encoding="utf-8")
    md_sha = sha256(md_path)

    bundle = BASE / "SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in (json_path, md_path, all_path, exceptional_path, catalog_path):
            zf.write(p, arcname=p.name)
        source_path = Path(globals().get("__file__", ""))
        if source_path.is_file():
            zf.write(source_path, arcname=source_path.name)

    print("\n" + "=" * 116)
    print("SU(4) EXCEPTIONAL ENUMERATOR STATUS: PASS")
    print("=" * 116)
    print(f"stable words                         : {len(stable_ordered):,}")
    print(f"SU(4) words                          : {len(su4_ordered):,}")
    print(f"exceptional-bearing words            : {len(exceptional_ordered):,}")
    print(f"new exceptional-only words           : {len(new_word_keys):,}")
    print(f"mixed words                          : {len(mixed_word_keys):,}")
    print(f"exceptional assignments              : {exceptional_assignment_count:,}")
    print(f"exceptional C-orbits                 : {exceptional_assignment_count // 2:,}")
    print(f"assignments with resolvent determinant: {assignments_with_resolvent_det:,}")
    print(f"final-only exceptional assignments   : {assignments_final_only:,}")
    print("final family histogram               :", dict(sorted(final_family_hist.items())))
    print("cut determinant histogram            :", dict(sorted(cut_det_hist.items())))
    print("JSON:", json_path, json_sha)
    print("MD:  ", md_path, md_sha)
    print("ALL: ", all_path, all_sha)
    print("EXC: ", exceptional_path, exceptional_sha)
    print("SIG: ", catalog_path, catalog_sha)
    print("ZIP: ", bundle, sha256(bundle))
    print("=" * 116)


if __name__ == "__main__":
    main()
